# Fraud Detection with Blind Insight Encrypted Data

This notebook demonstrates privacy-preserving fraud detection using Blind Insight's encrypted aggregation primitives. All computations are performed on encrypted data without exposing individual records.

## Prerequisites

1. Install required packages:
   ```bash
   pip install -r requirements.txt
   ```

2. Ensure the Blind Insight proxy and server are running

3. Make sure you have uploaded the fraud dataset to Blind Insight (10k records)

## Setup and Imports

We use the Blind Insight client to perform encrypted queries and aggregations on the fraud dataset.

In [6]:
import numpy as np
import warnings

# Suppress SSL warnings for local development
warnings.filterwarnings('ignore', message='Unverified HTTPS request')

# Import the Blind Insight client
import sys
sys.path.append('.')
from blind_insight_client import BlindInsightClient

## Configuration

Configure the Blind Insight connection and dataset parameters.

**Dataset**: Fraud analysis dataset with 10k records  
**Schema ID**: `i9yp3TutxNsCp3fgNWE4p2` (10k dataset with fixed schema indexes)

### Fraud Dataset Features
- **Target**: `is_fraud` (0 = legitimate, 1 = fraud)
- **Numeric features**: `amount`, `hour`, `device_risk_score`, `ip_risk_score`, `transaction_id`, `user_id`
- **One-hot encoded**: `country_*`, `merchant_category_*`, `transaction_type_*`

In [7]:
# Configuration
ORGANIZATION = "demo"
DATASET_SLUG = "fraud-analysis-training"
SCHEMA_SLUG = "fraud-analysis-schema"
API_URL = "https://proxy.local.blindinsight.io/"

USERNAME = "data_owner@localhost"
PASSWORD = "blindinsight"

# Use the 10k dataset schema (with fixed schema indexes)
SCHEMA_ID = "i9yp3TutxNsCp3fgNWE4p2"

# Initialize client
client = BlindInsightClient(
    api_url=API_URL,
    username=USERNAME,
    password=PASSWORD,
    verify_ssl=False
)

print(f"Client initialized for schema: {SCHEMA_ID}")

Client initialized for schema: i9yp3TutxNsCp3fgNWE4p2


## Helper Function

Define a helper to extract aggregation values from Blind Insight responses.

In [8]:
def agg_value(resp):
    """Extract aggregation value from Blind Insight response."""
    recs = resp.get("records", [])
    if not recs:
        return None  # No records found
    rec0 = recs[0]
    # Shape A: {records: [{data: {value: X}}]}
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else None
    # Shape B: {records: [{value: X}]}
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else None
    return None

print("Helper function defined: agg_value()")

Helper function defined: agg_value()


## Dataset Overview (Encrypted Counts)

Get an overview of the dataset using encrypted aggregations.

In [5]:
# Count total records using encrypted count on transaction_id
resp = client.aggregate(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    agg_filter="transaction_id:count(0~100000)",
    extra_filters=None,
    decrypt=False,
    schema_id=SCHEMA_ID
)
total_count = agg_value(resp)
print(f"Total records: {total_count:.0f}")

# Count fraud vs legitimate
for is_fraud in [0, 1]:
    resp = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter="transaction_id:count(0~100000)",
        extra_filters=[f"is_fraud:{is_fraud}"],
        decrypt=False,
        schema_id=SCHEMA_ID
    )
    count = agg_value(resp)
    label = "Fraud" if is_fraud else "Legitimate"
    pct = (count / total_count * 100) if total_count else 0
    print(f"{label} (is_fraud={is_fraud}): {count:.0f} ({pct:.1f}%)")

KeyboardInterrupt: 

## Mean Amount by Fraud Status

Compare mean transaction amounts between legitimate and fraudulent transactions using encrypted `avg`.

In [9]:
# Mean amount by fraud status
print("Mean transaction amount by fraud status:\n")

for is_fraud in [0, 1]:
    resp = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter="amount:avg(0~1000)",
        extra_filters=[f"is_fraud:{is_fraud}"],
        decrypt=False,
        schema_id=SCHEMA_ID
    )
    mean_val = agg_value(resp)
    label = "Fraud" if is_fraud else "Legitimate"
    if mean_val is not None:
        print(f"  {label}: ${mean_val:.2f}")
    else:
        print(f"  {label}: No data")

Mean transaction amount by fraud status:

  Legitimate: $9.00
  Fraud: $73.00


## Risk Scores by Fraud Status

Compare device_risk_score and ip_risk_score between fraud and legitimate transactions.

In [10]:
# Risk scores comparison
risk_features = ["device_risk_score", "ip_risk_score"]

print("Mean risk scores by fraud status:\n")

for feature in risk_features:
    print(f"{feature}:")
    for is_fraud in [0, 1]:
        resp = client.aggregate(
            organization=ORGANIZATION,
            dataset_slug=DATASET_SLUG,
            schema_slug=SCHEMA_SLUG,
            agg_filter=f"{feature}:avg(0~100)",
            extra_filters=[f"is_fraud:{is_fraud}"],
            decrypt=False,
            schema_id=SCHEMA_ID
        )
        mean_val = agg_value(resp)
        label = "Fraud" if is_fraud else "Legitimate"
        if mean_val is not None:
            print(f"  {label}: {mean_val:.2f}")
        else:
            print(f"  {label}: No data")
    print()

Mean risk scores by fraud status:

device_risk_score:
  Legitimate: 1.00
  Fraud: 8.00

ip_risk_score:
  Legitimate: 1.00
  Fraud: 8.00



## Transaction Type Analysis

Analyze fraud rates by transaction type (ATM, Online, POS, QR) using encrypted counts.

In [11]:
# Transaction type fraud rates
# The dataset has one-hot encoded transaction types: ATM, Online, POS, QR
transaction_types = ["ATM", "Online", "POS", "QR"]

print("Fraud count by transaction type:\n")

for tx_type in transaction_types:
    field = f"transaction_type_{tx_type}"
    # Count where this transaction type is 1 and is_fraud is 1
    resp = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter="transaction_id:count(0~100000)",
        extra_filters=[f"{field}:1", "is_fraud:1"],
        decrypt=False,
        schema_id=SCHEMA_ID
    )
    fraud_count = agg_value(resp) or 0
    
    # Total for this transaction type
    resp = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter="transaction_id:count(0~100000)",
        extra_filters=[f"{field}:1"],
        decrypt=False,
        schema_id=SCHEMA_ID
    )
    total_count = agg_value(resp) or 0
    
    if total_count > 0:
        fraud_rate = fraud_count / total_count * 100
        print(f"  {tx_type}: {fraud_count:.0f} fraud / {total_count:.0f} total ({fraud_rate:.1f}%)")
    else:
        print(f"  {tx_type}: No data")

Fraud count by transaction type:

  ATM: 95 fraud / 1867 total (5.1%)
  Online: 83 fraud / 1709 total (4.9%)
  POS: 65 fraud / 1835 total (3.5%)
  QR: 76 fraud / 1843 total (4.1%)


## Country Analysis

Analyze fraud distribution by country using encrypted aggregations.

In [12]:
# Country fraud rates
# The dataset has one-hot encoded countries: DE, FR, NG, TR, UK, US
countries = ["DE", "FR", "NG", "TR", "UK", "US"]

print("Fraud count by country:\n")

for country in countries:
    field = f"country_{country}"
    # Count where this country is 1 and is_fraud is 1
    resp = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter="transaction_id:count(0~100000)",
        extra_filters=[f"{field}:1", "is_fraud:1"],
        decrypt=False,
        schema_id=SCHEMA_ID
    )
    fraud_count = agg_value(resp) or 0
    
    # Total for this country
    resp = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter="transaction_id:count(0~100000)",
        extra_filters=[f"{field}:1"],
        decrypt=False,
        schema_id=SCHEMA_ID
    )
    total_count = agg_value(resp) or 0
    
    if total_count > 0:
        fraud_rate = fraud_count / total_count * 100
        print(f"  {country}: {fraud_count:.0f} fraud / {total_count:.0f} total ({fraud_rate:.1f}%)")
    else:
        print(f"  {country}: No data")

Fraud count by country:

  DE: 42 fraud / 1412 total (3.0%)
  FR: 49 fraud / 1489 total (3.3%)
  NG: 67 fraud / 67 total (100.0%)
  TR: 45 fraud / 1375 total (3.3%)
  UK: 54 fraud / 1424 total (3.8%)
  US: 62 fraud / 1487 total (4.2%)


## Merchant Category Analysis

Analyze fraud rates by merchant category.

In [13]:
# Merchant category fraud rates
# The dataset has one-hot encoded merchant categories: Clothing, Electronics, Food, Grocery, Travel
merchant_categories = ["Clothing", "Electronics", "Food", "Grocery", "Travel"]

print("Fraud count by merchant category:\n")

for category in merchant_categories:
    field = f"merchant_category_{category}"
    # Count where this category is 1 and is_fraud is 1
    resp = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter="transaction_id:count(0~100000)",
        extra_filters=[f"{field}:1", "is_fraud:1"],
        decrypt=False,
        schema_id=SCHEMA_ID
    )
    fraud_count = agg_value(resp) or 0
    
    # Total for this category
    resp = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter="transaction_id:count(0~100000)",
        extra_filters=[f"{field}:1"],
        decrypt=False,
        schema_id=SCHEMA_ID
    )
    total_count = agg_value(resp) or 0
    
    if total_count > 0:
        fraud_rate = fraud_count / total_count * 100
        print(f"  {category}: {fraud_count:.0f} fraud / {total_count:.0f} total ({fraud_rate:.1f}%)")
    else:
        print(f"  {category}: No data")

Fraud count by merchant category:

  Clothing: 75 fraud / 1433 total (5.2%)
  Electronics: 67 fraud / 1463 total (4.6%)
  Food: 49 fraud / 1463 total (3.3%)
  Grocery: 64 fraud / 1420 total (4.5%)
  Travel: 64 fraud / 1475 total (4.3%)


## Hour of Day Analysis

Analyze fraud patterns by hour of day using encrypted aggregations.

In [14]:
# Mean hour for fraud vs legitimate transactions
print("Mean transaction hour by fraud status:\n")

for is_fraud in [0, 1]:
    resp = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter="hour:avg(0~24)",
        extra_filters=[f"is_fraud:{is_fraud}"],
        decrypt=False,
        schema_id=SCHEMA_ID
    )
    mean_val = agg_value(resp)
    label = "Fraud" if is_fraud else "Legitimate"
    if mean_val is not None:
        print(f"  {label}: {mean_val:.1f}:00")
    else:
        print(f"  {label}: No data")

Mean transaction hour by fraud status:

  Legitimate: 0.0:00
  Fraud: 13.0:00


## Summary

This notebook demonstrates privacy-preserving fraud analysis using Blind Insight:

1. ✅ **Encrypted counts** - Count transactions without decrypting data
2. ✅ **Encrypted averages** - Compute mean values on encrypted fields
3. ✅ **Filtered aggregations** - Combine filters with aggregations (e.g., fraud rate by country)
4. ✅ **No plaintext exposure** - All computations performed on encrypted data

### Key Findings
- Compare fraud rates across transaction types, countries, and merchant categories
- Analyze risk scores (device and IP) by fraud status
- Identify high-risk transaction patterns without exposing individual records